# 🚀 NCKH 3 — Kaggle Thin Runner for TAWOS Planning-Time Snapshot (GitHub ➔ Kaggle ➔ Hugging Face Hub)

Notebook này đóng vai trò là **Thin Runner** chạy trên Kaggle GPU/CPU cho bộ thí nghiệm **13 dự án TAWOS Snapshot x 3 Setting (`final`, `cross`, `planning`)** (tích hợp từ `JohnnyPhan1712/SR-G3_planning-time-sp`):
1. **GitHub** lưu toàn bộ mã nguồn Python (`kaggle_TAWOS/`) trên branch thử nghiệm.
2. **Kaggle Runtime** đọc Kaggle Secrets (`hf_token` / `HF_TOKEN`), `git clone` đúng branch và chạy script chuẩn bị dữ liệu hoặc thực nghiệm.
3. **Hugging Face Hub** tự động nhận kết quả (`.parquet`, `.csv`, `.png`, `metadata.json`) vào `Snowflake06/NCKH3-StoryPoint-Artifact` trong **1 commit** mỗi lần chạy.

## ⚙️ Cell 1: Cấu hình Thực nghiệm TAWOS (Chỉnh sửa tham số tại đây)

In [ ]:
import os
import shlex
import subprocess

# ==============================================================================
# CHỈNH SỬA CẤU HÌNH CHO MỖI LẦN CHẠY TẠI ĐÂY
# ==============================================================================
GITHUB_REPO = "https://github.com/snowflake-vibing/NCKH_Johnny_Snowflake.git"
GIT_BRANCH = "main"  # Đổi thành tên branch nếu chạy thử nghiệm trên nhánh riêng

# Chọn 1 trong 4 chế độ:
# - "prepare_tawos_dataset" : Tạo/làm sạch 13 file <PROJECT>_snapshot.parquet từ TAWOS.sql(.zip) / parquet / MySQL và đẩy lên HF (CPU)
# - "tawos_baseline"        : Chạy Mean / Median / TF-IDF+Ridge x 13 dự án x 3 setting + Wilcoxon/A12/Holm (CPU)
# - "tawos_gpt2sp"          : Huấn luyện GPT2SP (Model Final + Model Planning) x 13 dự án x 3 setting + Auto-Resume (GPU)
# - "refresh_hf_artifact"   : Sinh README.md Dataset Card (YAML configs) + comparisons/ + datasets/ ngay trên NCKH3-StoryPoint-Artifact (CPU)
EXPERIMENT = "prepare_tawos_dataset"

HF_RESULTS_REPO = "NCKH3-StoryPoint-Artifact"
HF_CHECKPOINTS_REPO = "NCKH3-StoryPoint-Checkpoints"

# Ví dụ EXTRA_ARGS:
# - Với prepare_tawos_dataset: "--sql-path /kaggle/input/tawos-sql/TAWOS.sql"
# - Với tawos_baseline:        "--projects BAM,CLOV"
# - Với tawos_gpt2sp:          "--projects BAM,CLOV --epochs 20 --batch-size 32" (hoặc "--smoke-test")
EXTRA_ARGS = ""

print(f"Repo: {GITHUB_REPO} | Branch: {GIT_BRANCH} | Experiment: {EXPERIMENT}")

## 🔐 Cell 2: Kết nối Kaggle Secrets & Xác thực Hugging Face (Write Token)

In [ ]:
from huggingface_hub import HfApi

def load_hf_token_silently():
    for key in ("HF_TOKEN", "hf_token", "HUGGINGFACE_HUB_TOKEN"):
        val = os.environ.get(key)
        if val and val.strip():
            return val.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ("HF_TOKEN", "hf_token"):
            try:
                val = secrets.get_secret(key)
                if val and val.strip():
                    return val.strip()
            except Exception:
                continue
    except ImportError:
        pass
    return None

token = load_hf_token_silently()
if not token:
    raise RuntimeError(
        "Không tìm thấy HF_TOKEN hoặc hf_token trong Kaggle Secrets! "
        "Hãy vào Add-ons -> Secrets và bật token có quyền WRITE."
    )

os.environ["HF_TOKEN"] = token
api = HfApi(token=token)
HF_USERNAME = api.whoami()["name"]
print(f"✅ Đã xác thực Hugging Face thành công với tài khoản: {HF_USERNAME}")

## 📥 Cell 3: Clone / Cập nhật Mã nguồn từ GitHub Branch

In [ ]:
WORK_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_ROOT, "repo")

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", GIT_BRANCH], check=True)

os.chdir(REPO_DIR)
commit_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

import torch
gpu_desc = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"✅ Đã sẵn sàng tại {REPO_DIR} | Branch: {GIT_BRANCH} | Commit: {commit_hash} | Hardware: {gpu_desc}")

## ▶️ Cell 4: Cài đặt Thư viện Bổ sung (nếu cần) & Chạy Script Thực nghiệm

In [ ]:
import importlib.util
import sys

REQUIRED_PKGS = {
    "huggingface_hub": "huggingface_hub",
    "transformers": "transformers",
    "scipy": "scipy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing_pkgs = [pip_name for mod_name, pip_name in REQUIRED_PKGS.items() if importlib.util.find_spec(mod_name) is None]
if missing_pkgs:
    print("📦 Installing missing packages:", missing_pkgs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

EXPERIMENT_SCRIPTS = {
    "prepare_tawos_dataset": "kaggle_TAWOS/prepare_tawos_dataset.py",
    "tawos_baseline": "kaggle_TAWOS/run_tawos_baseline.py",
    "tawos_gpt2sp": "kaggle_TAWOS/run_tawos_gpt2sp.py",
    "refresh_hf_artifact": "scripts/migrate_hf_artifact.py",
}

if EXPERIMENT not in EXPERIMENT_SCRIPTS:
    raise ValueError(f"EXPERIMENT không hợp lệ: {EXPERIMENT}. Chọn: {list(EXPERIMENT_SCRIPTS.keys())}")

cmd = [sys.executable, "-u", EXPERIMENT_SCRIPTS[EXPERIMENT], "--results-repo", HF_RESULTS_REPO]
if EXPERIMENT == "tawos_gpt2sp":
    cmd.extend(["--checkpoints-repo", HF_CHECKPOINTS_REPO])
elif EXPERIMENT == "refresh_hf_artifact":
    cmd.append("--in-place")

if EXTRA_ARGS.strip():
    cmd.extend(shlex.split(EXTRA_ARGS))

print("🚀 Executing:", " ".join(cmd))
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Script exited with code {ret}")

## 📊 Cell 5: Hiển thị Tóm tắt Kết quả & Liên kết Hugging Face Hub

In [ ]:
import glob
import pandas as pd

outputs_root = os.path.join(WORK_ROOT, "outputs")
data_root = os.path.join(WORK_ROOT, "data")
csv_files = sorted(
    glob.glob(os.path.join(outputs_root, "**", "*.csv"), recursive=True)
    + glob.glob(os.path.join(data_root, "**", "*.csv"), recursive=True)
)
preferred = [p for p in csv_files if os.path.basename(p) in ("summary.csv", "dataset_summary.csv", "stat_tests_vs_median.csv", "wilcoxon.csv")]

print(f"📦 Hugging Face Results Dataset: https://huggingface.co/datasets/{HF_USERNAME}/{HF_RESULTS_REPO}")
if EXPERIMENT == "tawos_gpt2sp":
    print(f"🧠 Hugging Face Checkpoints Model: https://huggingface.co/{HF_USERNAME}/{HF_CHECKPOINTS_REPO}")

if preferred:
    for target_csv in preferred[-2:]:
        print(f"\n📄 Preview ({target_csv}):")
        df_show = pd.read_csv(target_csv)
        try:
            display(df_show)
        except NameError:
            print(df_show.to_string(index=False))
else:
    print("ℹ️ Không tìm thấy file summary CSV cục bộ; kiểm tra trực tiếp trên HF Dataset Repo.")